In [29]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [38]:


# ==========================================
# 1. КОНФИГУРАЦИЯ И СТИЛИ (Меняем все тут)
# ==========================================
DURATION = 540

# Словарь с визуальными настройками
STYLE = {
    'tc':   dict(color='#636EFA', name='Транскритическая'),
    'flip': dict(color='#EF553B', name='Flip'),
    'ns':   dict(color='#00CC96', name='Неймарк–Сакер'),
    'curr': dict(color='red', size=12, name='Текущий параметр'),
    'phase_line': dict(color="rgba(180,180,180,0.4)"),
    'phase_marker': dict(size=6, colorscale="Viridis"),
}

# Шаблон подсказки
HOVER_TMPL = "x: %{x:.3f}<br>y: %{y:.3f}<br>Итерация: %{customdata}<extra></extra>"

# ==========================================
# 2. РАСЧЕТЫ (Кэширование)
# ==========================================
tilda_values = np.linspace(0, 280, 70)
u2 = 0.6
u1 = 0.97
v = 0.5

# Бифуркационные кривые
u1_grid = np.linspace(0.01, 0.99, 500)
den = (1 - u1_grid) * (1 - u2)
alpha_tc = (1 - v * (1 - u2)) / den
alpha_flip = (1 - 3 * v * (1 - u2)) / den
alpha_ns = (3 - 2 * v * (1 - u2)) / den

cache = []
for t in tilda_values:
    # Фазовый портрет
    x0, y0 = 0.3, 0.6
    x_n, y_n = x0, y0
    xs, ys, it = [], [], []
    n = 100
    for i in range(n):
        next_x = t * y_n * (1 - y_n) * (1 - u1)
        next_y = (x_n + v * y_n)*(1-u2)
        xs.append(next_x)
        ys.append(next_y)
        it.append(i)
        x_n, y_n = next_x, next_y

    cache.append({
        "tilda_alpha": t,
        "u1_grid": u1_grid,
        "alpha_tc": alpha_tc,
        "alpha_flip": alpha_flip,
        "alpha_ns": alpha_ns,
        "phase": pd.DataFrame({"x": xs, "y": ys, "iteration": it})
    })

# Вычисляем глобальные границы (один раз для всех графиков)
all_x = np.concatenate([c["phase"]["x"] for c in cache])
all_y = np.concatenate([c["phase"]["y"] for c in cache])
g_xmax, g_ymax = all_x.max(), all_y.max()
# (Можно добавить min, если нужно, но у вас от 0)
GLOBAL_X_RANGE = [0, max(1.0, g_xmax * 1.05)]
GLOBAL_Y_RANGE = [0, max(1.0, g_ymax * 1.05)]


# ==========================================
# 3. ФУНКЦИИ-ГЕНЕРАТОРЫ ГРАФИКОВ
# ==========================================

def get_bifurcation_traces(data, current_u1, current_alpha, showlegend=True):
    """Возвращает список из 4 трейсов: 3 линии бифуркаций + 1 точка параметра"""
    return [
        go.Scatter(x=data["u1_grid"], y=data["alpha_tc"], mode='lines',
                   line=dict(color=STYLE['tc']['color']), name=STYLE['tc']['name'], showlegend=showlegend),

        go.Scatter(x=data["u1_grid"], y=data["alpha_flip"], mode='lines',
                   line=dict(color=STYLE['flip']['color']), name=STYLE['flip']['name'], showlegend=showlegend),

        go.Scatter(x=data["u1_grid"], y=data["alpha_ns"], mode='lines',
                   line=dict(color=STYLE['ns']['color']), name=STYLE['ns']['name'], showlegend=showlegend),

        go.Scatter(x=[current_u1], y=[current_alpha], mode='markers',
                   marker=dict(color=STYLE['curr']['color'], size=STYLE['curr']['size']),
                   name=STYLE['curr']['name'], showlegend=showlegend)
    ]

def get_phase_traces(phase_df, showlegend=False):
    """Возвращает список из 2 трейсов: Маркеры + Линия"""
    return [
        # 1. Маркеры (с данными об итерации)
        go.Scatter(
            x=phase_df["x"], y=phase_df["y"], mode='markers',
            marker=dict(size=STYLE['phase_marker']['size'], color=phase_df["iteration"], colorscale=STYLE['phase_marker']['colorscale']),
            customdata=phase_df["iteration"],
            hovertemplate=HOVER_TMPL,
            showlegend=showlegend
        ),
        # 2. Линия (полупрозрачная)
        go.Scatter(
            x=phase_df["x"], y=phase_df["y"], mode='lines',
            line=dict(color=STYLE['phase_line']['color']),
            showlegend=showlegend
        )
    ]

# ==========================================
# 4. СОЗДАНИЕ ГЛАВНОЙ АНИМАЦИИ
# ==========================================

fig = make_subplots(rows=1, cols=2, subplot_titles=("Бифуркации", "Фазовый портрет"))

# -- Инициализация (Первый кадр) --
first = cache[0]
# Добавляем слева (бифуркации)
for trace in get_bifurcation_traces(first, u1, first["tilda_alpha"], showlegend=True):
    fig.add_trace(trace, row=1, col=1)

# Добавляем справа (фазовый)
for trace in get_phase_traces(first["phase"], showlegend=False):
    fig.add_trace(trace, row=1, col=2)

# -- Создание кадров --
frames = []
for c in cache:
    # Собираем все трейсы в один плоский список для кадра
    # Порядок ВАЖЕН! Он должен совпадать с порядком добавления в fig (4 слева + 2 справа)
    frame_traces = (
        get_bifurcation_traces(c, u1, c["tilda_alpha"], showlegend=False) +
        get_phase_traces(c["phase"], showlegend=False)
    )

    frames.append(go.Frame(
        data=frame_traces,
        name=f"{c['tilda_alpha']:.3f}",
        layout=go.Layout(
            title=dict(text=f"Текущее значение: u1 = {u1:.3f},  ~α = {c['tilda_alpha']:.3f}"),
            xaxis2=dict(range=GLOBAL_X_RANGE), # Используем глобальные границы
            yaxis2=dict(range=GLOBAL_Y_RANGE)
        )
    ))

fig.frames = frames

# -- Финальные настройки Layout --
fig.update_xaxes(title_text="u₁", row=1, col=1)
fig.update_yaxes(title_text="~α", row=1, col=1)
fig.update_xaxes(title_text="x", row=1, col=2)
fig.update_yaxes(title_text="y", row=1, col=2)

fig.update_layout(
    legend=dict(x=0, xanchor='left', y=1, bgcolor='rgba(0,0,0,0)'),
    width=1400, height=450,
    updatemenus=[dict(type="buttons", showactive=False,
                      buttons=[dict(label="Play", method="animate",
                                    args=[None, {"frame": {"duration": DURATION, "redraw": True}, "fromcurrent": True}]),
                               dict(label="Pause", method="animate",
                                    args=[[None], {"frame": {"duration": 0}, "mode": "immediate"}])])]
)

fig.show()

# ==========================================
# 5. ВЫВОД СТАТИЧЕСКИХ КАДРОВ
# ==========================================

def show_static_frame_optimized(index):
    data = cache[index]
    curr_alpha = data["tilda_alpha"]

    static_fig = make_subplots(rows=1, cols=2, subplot_titles=("Бифуркации", "Фазовый портрет"))

    # Слева
    for trace in get_bifurcation_traces(data, u1, curr_alpha, showlegend=True):
        static_fig.add_trace(trace, row=1, col=1)

    # Справа
    for trace in get_phase_traces(data["phase"], showlegend=False):
        static_fig.add_trace(trace, row=1, col=2)

    # Настройки
    static_fig.update_layout(
        height=400, width=1000,
        title_text=f"Кадр {index}: u1 = {u1:.3f}, ~α = {curr_alpha:.3f}",
        showlegend=True
    )
    # Оси
    static_fig.update_xaxes(title_text="u1", row=1, col=1)
    static_fig.update_yaxes(title_text="~α", row=1, col=1)
    static_fig.update_xaxes(title_text="x", range=GLOBAL_X_RANGE, row=1, col=2)
    static_fig.update_yaxes(title_text="y", range=GLOBAL_Y_RANGE, row=1, col=2)

    static_fig.show()

# Вывод
print("Отображение статичных кадров")
for i in range(0, len(cache), 10):
    show_static_frame_optimized(i)

Отображение статичных кадров...
